# Figure 4a — Migration Contribution Map

This notebook reproduces panel A from the main submission Figure 4 as a standalone figure, reformatted for a 115 mm x 85 mm output size.

In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.font_manager import FontProperties

DATA_FOLDER = Path('../01_data/')
FIGURE_FOLDER = Path('../03_documents/01_main_figures/')
os.makedirs(FIGURE_FOLDER, exist_ok=True)

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans Mono'
mpl.rcParams['font.size'] = 8
sns.set(style='whitegrid')

## Load and Prepare Data

In [ ]:
gudd_change_path = DATA_FOLDER / '04_final_demographic_data/01_static_boundaries/gudd_change_2000_2020_static_boundaries.csv'
world_shp_path = DATA_FOLDER / '02_auxiliary_data/03_mapping/ne_50m_admin_0_countries.shp'

change_data = pd.read_csv(gudd_change_path)
world = gpd.read_file(world_shp_path)

change_data_filtered = change_data[
    (change_data['YearOfBirth'] <= 2000) &
    (change_data['YearOfDeath'] >= 2020)
].copy()

for col in [
    'longitude', 'latitude', 'total_pop_2020', 'total_pop_Delta',
    'total_migration', 'natural_change', 'perc_from_migration'
]:
    change_data_filtered[col] = pd.to_numeric(change_data_filtered[col], errors='coerce')

migration_map_data = change_data_filtered.copy()
migration_map_data = migration_map_data[
    (migration_map_data['total_migration'] > 0) &
    (migration_map_data['natural_change'] > 0)
].copy()

migration_map_data = migration_map_data.dropna(
    subset=['longitude', 'latitude', 'total_pop_Delta', 'perc_from_migration']
)

print(f'Filtered migration map rows: {migration_map_data.shape[0]:,}')

## Figure 4a Styling

In [ ]:
mono_font = FontProperties(family='DejaVu Sans Mono', size=8)
small_font = FontProperties(family='DejaVu Sans Mono', size=6)

migration_colors = [
    '#fdd070', '#fdae61', '#f98e52', '#f46d43', '#e34a33', '#d73027',
    '#c51b7d', '#ae017e', '#8c0273', '#5f0165', '#2b0040'
]
migration_cmap = mcolors.LinearSegmentedColormap.from_list('migration', migration_colors)
norm_migration = mpl.colors.Normalize(vmin=0, vmax=100)

MM_PER_INCH = 25.4
FIGURE_SIZE_MM = (115, 85)
FIGURE_SIZE_IN = tuple(value / MM_PER_INCH for value in FIGURE_SIZE_MM)

output_path = FIGURE_FOLDER / 'fig4a_migration_map_115x85mm.pdf'

## Plot Standalone Panel

In [ ]:
fig, ax_map = plt.subplots(figsize=FIGURE_SIZE_IN)

sizes = np.interp(
    migration_map_data['total_pop_Delta'],
    (migration_map_data['total_pop_Delta'].min(), migration_map_data['total_pop_Delta'].max()),
    (2, 800),
)

# Match main Figure 4a: plot points first, then draw the world layer beneath via z-order.
ax_map.scatter(
    migration_map_data.longitude,
    migration_map_data.latitude,
    c=migration_map_data.perc_from_migration,
    cmap=migration_cmap,
    norm=norm_migration,
    s=sizes,
    edgecolor='black',
    linewidth=0,
    alpha=0.5,
    zorder=2,
)
world.plot(ax=ax_map, color='#E6E6E6', edgecolor='0.5', linewidth=0.3, zorder=1)

ax_map.set_xlim(-180, 180)
ax_map.set_ylim(-55, 90)
ax_map.set_xticks([])
ax_map.set_yticks([])
for spine in ax_map.spines.values():
    spine.set_visible(False)

sm_map = plt.cm.ScalarMappable(cmap=migration_cmap, norm=norm_migration)
sm_map.set_array([])
cbar = fig.colorbar(
    sm_map,
    ax=ax_map,
    orientation='horizontal',
    fraction=0.08,
    pad=0.08,
    aspect=40,
)
cbar.set_label('% population growth from migration (2000-2020)', fontproperties=mono_font)
for tick in cbar.ax.get_xticklabels():
    tick.set_fontproperties(mono_font)
    tick.set_fontsize(7)
cbar.ax.text(
    0.0, 1.2, 'Pop. \u0394 driven\nby natural change',
    ha='left', va='bottom', transform=cbar.ax.transAxes,
    fontproperties=small_font,
)
cbar.ax.text(
    1.0, 1.2, 'Pop. \u0394 driven\nby migration',
    ha='right', va='bottom', transform=cbar.ax.transAxes,
    fontproperties=small_font,
)

fig.subplots_adjust(left=0.025, right=0.975, top=0.99, bottom=0.18)
fig.savefig(output_path, format='pdf', dpi=300)
print(f'Saved: {output_path}')
plt.show()